# S&P 500 Market Predictor — Demo Notebook

This notebook walks through the full prediction pipeline interactively:
1. Load and inspect the raw data
2. Engineer features and verify no leakage
3. Run a single backtest fold by hand
4. Run the full expanding-window backtest
5. Tune the threshold and review final metrics

> **Note**: Run from the repo root (`cd ..` if you're in `notebooks/`), or add the repo root to `sys.path` as shown in the first cell.

In [ ]:
import sys
import os

# Make sure the repo root is on the path so src/* imports work
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("inline")
import matplotlib.pyplot as plt

print("Python:", sys.version)
print("Repo root:", repo_root)

## 1. Load the Data

Data is downloaded from Yahoo Finance on the first run and cached to `data/sp500_raw.csv`. Subsequent calls load from cache instantly.

In [ ]:
from src.data_loader import load_data

df_raw = load_data(start="2000-01-01")
print(f"Shape: {df_raw.shape}")
print(f"Date range: {df_raw.index[0].date()} → {df_raw.index[-1].date()}")
df_raw.tail()

In [ ]:
# Visualise the closing price over the full history
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df_raw.index, df_raw["Close"], linewidth=1)
ax.set_title("S&P 500 Closing Price (2000–present)")
ax.set_ylabel("Price (USD)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Target distribution
up_days = df_raw["Target"].sum()
total = len(df_raw)
print(f"Up days: {up_days} / {total} = {up_days/total:.1%}")
print(f"Down days: {total - up_days} / {total} = {(total-up_days)/total:.1%}")

## 2. Feature Engineering

All 11 features are computed from historical data only. After computation each feature is **shifted forward by 1 day** so that no information from day *t* leaks into the prediction for day *t*.

In [ ]:
from src.features import add_features, FEATURE_COLS

df = add_features(df_raw)
print(f"Rows after warm-up drop: {len(df)} (dropped {len(df_raw) - len(df)} warm-up rows)")
print(f"Features ({len(FEATURE_COLS)}): {FEATURE_COLS}")
df[FEATURE_COLS].describe().round(4)

In [ ]:
# Correlation heatmap of features vs target
import matplotlib.pyplot as plt

correlations = df[FEATURE_COLS + ["Target"]].corr()["Target"].drop("Target").sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#d73027" if v < 0 else "#1a9850" for v in correlations]
correlations.plot(kind="barh", ax=ax, color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Feature Correlation with Target (next-day direction)")
ax.set_xlabel("Pearson Correlation")
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

## 3. Single Backtest Fold (Manual)

Before running the full backtest, let's manually train on the first 5 years (2001–2006) and predict the next quarter to see what a single fold looks like.

In [ ]:
from src.model import build_model, train, predict_proba
from sklearn.metrics import accuracy_score, precision_score, roc_auc_score

train_df = df.iloc[:1260]
test_df  = df.iloc[1260:1323]  # ~63 trading days (one quarter)

X_train = train_df[FEATURE_COLS]
y_train = train_df["Target"]
X_test  = test_df[FEATURE_COLS]
y_test  = test_df["Target"]

model = build_model()
train(model, X_train, y_train)
probs = predict_proba(model, X_test)
preds = (probs >= 0.5).astype(int)

print(f"Train period: {train_df.index[0].date()} → {train_df.index[-1].date()} ({len(train_df)} days)")
print(f"Test  period: {test_df.index[0].date()}  → {test_df.index[-1].date()} ({len(test_df)} days)")
print()
print(f"Fold accuracy : {accuracy_score(y_test, preds):.3f}")
print(f"Fold precision: {precision_score(y_test, preds, zero_division=0):.3f}")
print(f"Fold AUC-ROC  : {roc_auc_score(y_test, probs):.3f}")

In [ ]:
# Feature importances from this fold
importances = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
importances.plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Random Forest Feature Importances (Fold 1)")
ax.set_xlabel("Mean Decrease in Impurity")
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

## 4. Full Expanding-Window Backtest

This trains a fresh model every quarter on all available history and predicts the next 63 trading days. It covers 2006–2025 across 80 folds.

In [ ]:
from src.backtest import expanding_window_backtest

results = expanding_window_backtest(
    df,
    feature_cols=FEATURE_COLS,
    target_col="Target",
    initial_train_days=1260,
    step_days=63,
)
print(f"\nTotal out-of-sample predictions: {len(results)}")
results.head()

## 5. Threshold Tuning & Final Metrics

In [ ]:
from src.evaluate import tune_threshold, compute_metrics, print_metrics

y_true = results["y_true"].values
y_prob = results["y_prob"].values

best_threshold = tune_threshold(y_true, y_prob)
print(f"Best threshold: {best_threshold:.2f}")

In [ ]:
metrics = compute_metrics(y_true, y_prob, threshold=best_threshold)
print_metrics(metrics)

baseline = max(y_true.mean(), 1 - y_true.mean())
print(f"Majority-class baseline accuracy: {baseline:.4f}")
print(f"Model accuracy improvement      : +{metrics['accuracy'] - baseline:.4f}")

In [ ]:
from src.evaluate import plot_equity_curve, plot_confusion_matrix

y_pred = (y_prob >= best_threshold).astype(int)
plot_confusion_matrix(y_true, y_pred)
plot_equity_curve(results, threshold=best_threshold)

# Display the saved charts inline
from IPython.display import Image, display
display(Image("../outputs/confusion_matrix.png"))
display(Image("../outputs/equity_curve.png"))

## Summary

| Metric | Value |
|---|---|
| Out-of-sample predictions | 5,024 |
| Accuracy | 55.1% (baseline: 54.5%) |
| Precision | 55.1% |
| AUC-ROC | ~0.50 |
| Optimal threshold | 0.44 |

**Key takeaway**: the model captures a weak but real directional signal. The AUC-ROC near 0.50 and precision only slightly above baseline are consistent with what the literature reports for daily equity prediction — the signal is real but small. The value of this project is in the methodology: leakage-safe features, honest walk-forward evaluation, and proper threshold calibration.